## Read the first few lines of the Amazon Snap File to understand the structure

In [1]:
import pandas as pd
import gzip
from urllib.request import urlopen
import io

url = "https://snap.stanford.edu/data/bigdata/amazon/amazon-meta.txt.gz"

with urlopen(url) as response:
#with gzip.open(file_path, 'rt', encoding='latin-1') as f:
    with gzip.open(response, mode='rt', encoding='latin-1') as f:
        for i in range(22):
            print(f.readline().strip())

# Full information about Amazon Share the Love products
Total items: 548552

Id:   0
ASIN: 0771044445
discontinued product

Id:   1
ASIN: 0827229534
title: Patterns of Preaching: A Sermon Sampler
group: Book
salesrank: 396585
similar: 5  0804215715  156101074X  0687023955  0687074231  082721619X
categories: 2
|Books[283155]|Subjects[1000]|Religion & Spirituality[22]|Christianity[12290]|Clergy[12360]|Preaching[12368]
|Books[283155]|Subjects[1000]|Religion & Spirituality[22]|Christianity[12290]|Clergy[12360]|Sermons[12370]
reviews: total: 2  downloaded: 2  avg rating: 5
2000-7-28  cutomer: A2JW67OY8U6HHK  rating: 5  votes:  10  helpful:   9
2003-12-14  cutomer: A2VE83MZF98ITY  rating: 5  votes:   6  helpful:   5

Id:   2
ASIN: 0738700797


## Download the data from SNAP and store in a dataframe and save to a Parquet file with compression

In [2]:
import pandas as pd
import gzip
from urllib.request import urlopen
import io

url = "https://snap.stanford.edu/data/bigdata/amazon/amazon-meta.txt.gz"
#file_path = r"../data/amazon-meta.txt.gz"


products = []
current_item = {}

with urlopen(url) as response:
#with gzip.open(file_path, 'rt', encoding='latin-1') as f:
    with gzip.open(response, mode='rt', encoding='latin-1') as f:
        for line in f:
            line = line.strip()

            if line.startswith("Id:"):
                if current_item:
                    products.append(current_item)
                current_item = {'Id': line.split(":")[1].strip(),'Active': 'Y'}
            elif line.startswith("discontinued"):
                current_item['Active'] = 'N'
            elif line.startswith("ASIN"):
                current_item['ASIN'] = line.split(":")[1].strip()
            elif line.startswith("title"):
                current_item['Title'] = line.split(":", 1)[1].strip()
            elif line.startswith("group"):
                current_item['Group'] = line.split(":")[1].strip()
            elif line.startswith("salesrank"):
                current_item['Salesrank'] = line.split(":")[1].strip()
            elif line.startswith('similar:'):
                parts = line.split()  # Splits by any whitespace
                current_item["similarCount"] = parts[1]
                current_item['SimilarASINs'] = parts[2:]
            elif line.startswith('|'):
                if 'Categories' not in current_item:
                    current_item['Categories'] = []
                    current_item['CleanCategories'] = []
                current_item['Categories'].append(line)
                # 4 levels
                line_clean = [cat.split('[')[0] for cat in line.strip('|').split('|')][:4]
                current_item['CleanCategories'].append(" > ".join(line_clean))
            elif line.startswith('reviews:'):
                parts = line.split()
                try:
                    current_item['reviews_count'] = int(parts[2])
                    current_item['avg_rating'] = float(parts[-1])
                except (ValueError, IndexError):
                    current_item['reviews_count'] = 0
                    current_item['avg_rating'] = 0.0
        if current_item:
            products.append(current_item)

df = pd.DataFrame(products)
df.to_parquet('../data/amazon_data.parquet', compression='gzip')
print(df.head())

  Id Active        ASIN                                              Title  \
0  0      N  0771044445                                                NaN   
1  1      Y  0827229534            Patterns of Preaching: A Sermon Sampler   
2  2      Y  0738700797                         Candlemas: Feast of Flames   
3  3      Y  0486287785   World War II Allied Fighter Planes Trading Cards   
4  4      Y  0842328327  Life Application Bible Commentary: 1 and 2 Tim...   

  Group Salesrank similarCount  \
0   NaN       NaN          NaN   
1  Book    396585            5   
2  Book    168596            5   
3  Book   1270652            0   
4  Book    631289            5   

                                        SimilarASINs  \
0                                                NaN   
1  [0804215715, 156101074X, 0687023955, 068707423...   
2  [0738700827, 1567184960, 1567182836, 073870052...   
3                                                 []   
4  [0842328130, 0830818138, 0842330313, 084232

## Read the Parquet File and load to a Dataframe

In [3]:
# pip install fastparquet
import pandas as pd

file_path = r"../data/amazon_data.parquet"
df = pd.read_parquet(file_path)
print(df.head())

print("Data loaded successfully!")

  Id Active        ASIN                                              Title  \
0  0      N  0771044445                                               None   
1  1      Y  0827229534            Patterns of Preaching: A Sermon Sampler   
2  2      Y  0738700797                         Candlemas: Feast of Flames   
3  3      Y  0486287785   World War II Allied Fighter Planes Trading Cards   
4  4      Y  0842328327  Life Application Bible Commentary: 1 and 2 Tim...   

  Group Salesrank similarCount  \
0  None      None         None   
1  Book    396585            5   
2  Book    168596            5   
3  Book   1270652            0   
4  Book    631289            5   

                                        SimilarASINs  \
0                                               None   
1  [0804215715, 156101074X, 0687023955, 068707423...   
2  [0738700827, 1567184960, 1567182836, 073870052...   
3                                                 []   
4  [0842328130, 0830818138, 0842330313, 084232

In [4]:
print("Number of Products (Rows):", len(df))

Number of Products (Rows): 548552


## Reduce to 5 core (Each nodes must have k neighbors: k = 5)

`Create a graph using the SimilarASINs. The number of products having entry is 548552. But the graph identifies ASINs that are not having any details (ghost nodes). These are dropped later since they dont have any features.Also merge the edges that have (a,b) and (b,a) and create unidirected graph`

In [5]:
import pandas as pd
import networkx as nx

# Create all unique edges (Undirected)
edges = set()
for idx, row in df.iterrows():
    u = row['ASIN']
    if isinstance(row['SimilarASINs'], list):
        for v in row['SimilarASINs']:
            # Sort to ensure (A, B) and (B, A) are treated as the same undirected edge
            # Reduces the edges from 1.78M to 1.54 M
            edge = tuple(sorted((u, v)))
            edges.add(edge)

# Build the graph
G = nx.Graph()
G.add_edges_from(edges)


In [6]:
print(f"Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

Graph has 554789 nodes and 1545228 edges.


`Identify the 5-core and perform LCC filterning`
https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.core.k_core.html

In [7]:
# Extract the 5 core (Iterative prune until all nodes have degree >= 5)
G_FiveCore = nx.k_core(G, k=5)

# Extract the Largest Connected Component 
lcc_nodes = max(nx.connected_components(G_FiveCore), key=len)
G_final = G_FiveCore.subgraph(lcc_nodes).copy()

print(f"Nodes in Final Graph: {G_final.number_of_nodes()}")


Nodes in Final Graph: 111624


`Print the Group wise node count`

In [8]:
df_lcc = df[df['ASIN'].isin(G_final.nodes())].copy()
print(df_lcc['Group'].value_counts())

Group
Book     78311
Music     9890
DVD       9019
Video     7008
Name: count, dtype: int64


In [9]:
import networkx as nx

# Identify valid ASINs (Active 'Y' AND has a valid Title)
valid_nodes_df = df_lcc[
    (df_lcc['Active'] == 'Y') & 
    (df_lcc['Title'].notna()) & 
    (df_lcc['Title'] != "None")&
    (df_lcc['avg_rating'] > 0)
].copy()

valid_asins = set(valid_nodes_df['ASIN'])

# Create a subgraph containing ONLY these valid nodes
G_valid = G_final.subgraph(valid_asins).copy()

# Re-run 5-Core Pruning
# Some nodes will drop out because their neighbors were removed in step 1
G_FiveCore_Final = nx.k_core(G_valid, k=5)

# Extract the final Largest Connected Component (LCC)
lcc_nodes_final = max(nx.connected_components(G_FiveCore_Final), key=len)
G_final_clean = G_FiveCore_Final.subgraph(lcc_nodes_final).copy()

print(f"Final Research Nodes: {G_final_clean.number_of_nodes()}")


Final Research Nodes: 24492


In [10]:
final_asins = list(G_final_clean.nodes())
df_active = df[df['ASIN'].isin(final_asins)].copy()
print(f"Columns in df_active: {df_active.columns.tolist()}")

Columns in df_active: ['Id', 'Active', 'ASIN', 'Title', 'Group', 'Salesrank', 'similarCount', 'SimilarASINs', 'Categories', 'CleanCategories', 'reviews_count', 'avg_rating']


# Unique Categories

In [11]:
all_paths = df_active['CleanCategories'].explode().dropna().unique()
print(f"Total Unique Categories: {len(all_paths)}")

category_to_id = {name: i for i, name in enumerate(all_paths)}

def map_to_ids(cat_list):
    if isinstance(cat_list, list):
        return list(set(category_to_id[cat] for cat in cat_list if cat in category_to_id))
    return []

df_active['CategoryIDs'] = df_active['CleanCategories'].apply(map_to_ids)

print(df_active[['CleanCategories', 'CategoryIDs']].head())

Total Unique Categories: 966
                                      CleanCategories  \
2   [Books > Subjects > Religion & Spirituality > ...   
10  [Books > Subjects > Literature & Fiction > His...   
21  [ > DVD > Genres > Drama,  > DVD > Genres > Sc...   
43  [Books > Subjects > Literature & Fiction > Wor...   
82  [Books > Subjects > Literature & Fiction > Wor...   

                                          CategoryIDs  
2                                                 [0]  
10                                          [1, 2, 3]  
21  [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16,...  
43                                       [20, 21, 22]  
82                                       [20, 22, 23]  


In [12]:
# Range of Salesrank

rank_min = df_active['Salesrank'].min()
rank_max = df_active['Salesrank'].max()

print(f"Salesrank Range: {rank_min} to {rank_max}")

print(df_active['Salesrank'].describe())

Salesrank Range: 0 to 99997
count     24492
unique    22778
top        5926
freq          6
Name: Salesrank, dtype: object


# Add New Id field

In [13]:
df_active['New_Id'] = range(len(df_active))

cols = ['New_Id'] + [c for c in df_active.columns if c != 'New_Id']
df_active = df_active[cols]
print(df_active[['New_Id', 'Id', 'ASIN', 'CategoryIDs']].head())

    New_Id  Id        ASIN                                        CategoryIDs
2        0   2  0738700797                                                [0]
10       1  10  0375709363                                          [1, 2, 3]
21       2  21  0790747324  [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16,...
43       3  43  0486268780                                       [20, 21, 22]
82       4  82  0140430407                                       [20, 22, 23]


In [14]:
df_final = df_active[['New_Id', 'Title', 'Group', 'Salesrank', 'reviews_count', 'CategoryIDs']].copy()
df_final['Salesrank'] = pd.to_numeric(df_final['Salesrank'], errors='coerce').fillna(0).astype(int)
df_final['reviews_count'] = df_final['reviews_count'].astype(int)
print(df_final.head())


    New_Id                                              Title Group  \
2        0                         Candlemas: Feast of Flames  Book   
10       1                             The Edward Said Reader  Book   
21       2                                   The Time Machine   DVD   
43       3             Selected Poems (Dover Thrift Editions)  Book   
82       4  Puddnhead Wilson : And, Those Extraordinary Tw...  Book   

    Salesrank  reviews_count  \
2      168596             12   
10     220379              6   
21        795            140   
43     260300              2   
82      60668              4   

                                          CategoryIDs  
2                                                 [0]  
10                                          [1, 2, 3]  
21  [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16,...  
43                                       [20, 21, 22]  
82                                       [20, 22, 23]  


# Encode Title

## FastText

In [60]:
import fasttext

In [61]:
model = fasttext.load_model('/home/hice1/pmaji3/scratch/gnn02/cc.en.300.bin')

In [62]:
titles = df_final['Title'].tolist()
embeddings = [model.get_sentence_vector(text) for text in titles]

In [63]:
df_final['Title_Embeddings'] = list(embeddings)
print(df_final[['Title', 'Title_Embeddings']].head())
print(df_final['Title_Embeddings'].size, df_final['Title_Embeddings'].iloc[0].shape)

                                                Title  \
2                          Candlemas: Feast of Flames   
10                             The Edward Said Reader   
21                                   The Time Machine   
43             Selected Poems (Dover Thrift Editions)   
82  Puddnhead Wilson : And, Those Extraordinary Tw...   

                                     Title_Embeddings  
2   [-0.030154344, 0.0013526536, 0.01520529, -0.03...  
10  [-0.027259978, 0.07363535, 0.032825436, -0.007...  
21  [-0.01067525, 0.08308516, -0.03514494, -0.0338...  
43  [0.014430257, 0.049291063, -0.017266652, 0.034...  
82  [-0.021529263, 0.036200337, 0.0067345924, -0.0...  
24492 (300,)


In [64]:
# Extract the Embedding to a csv .. TODO: Other features to be appended
embeddings_df = pd.DataFrame(df_final['Title_Embeddings'].tolist())
embeddings_df.to_csv('../data/node_features.csv', index=False, header=False)

In [65]:
import os

file_path = '../data/node_features.csv'
size_bytes = os.path.getsize(file_path)
size_mb = size_bytes / (1024 * 1024)

print(f"File Size: {size_mb:.2f} MB")

File Size: 87.19 MB


## SBERT

In [31]:
# pip install -U sentence-transformers
from sentence_transformers import SentenceTransformer, util

/home/hice1/pmaji3/scratch/gnn02/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [32]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df_final['Title'].tolist(), show_progress_bar=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 548.70it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 766/766 [00:12<00:00, 61.41it/s] 


In [33]:
df_final['Title_Embeddings'] = list(embeddings)
print(df_final[['Title', 'Title_Embeddings']].head())
print(df_final['Title_Embeddings'].size, df_final['Title_Embeddings'].iloc[0].shape)

                                                Title  \
2                          Candlemas: Feast of Flames   
10                             The Edward Said Reader   
21                                   The Time Machine   
43             Selected Poems (Dover Thrift Editions)   
82  Puddnhead Wilson : And, Those Extraordinary Tw...   

                                     Title_Embeddings  
2   [-0.02202112, 0.13937773, 0.02322836, 0.038774...  
10  [-0.04886586, 0.01876081, -0.007205362, 0.0351...  
21  [-0.12326239, 0.04113713, 0.021665249, 0.00582...  
43  [-0.003108027, -0.03348946, 0.074621305, -0.02...  
82  [-0.06499201, -0.09111721, -0.030142087, 0.012...  
24492 (384,)


In [34]:
# Extract the Embedding to a csv .. TODO: Other features to be appended
embeddings_df = pd.DataFrame(df_final['Title_Embeddings'].tolist())
embeddings_df.to_csv('../data/node_features.csv', index=False, header=False)

In [35]:
import os

file_path = '../data/node_features.csv'
size_bytes = os.path.getsize(file_path)
size_mb = size_bytes / (1024 * 1024)

print(f"File Size: {size_mb:.2f} MB")

File Size: 109.74 MB


# Generate the Edge Index

In [36]:
asin_to_id = pd.Series(df_active.New_Id.values, index=df_active.ASIN).to_dict()
edges = df_active[['New_Id', 'SimilarASINs']].explode('SimilarASINs')
edges['Target_Id'] = edges['SimilarASINs'].map(asin_to_id)
edge_index_df = edges.dropna(subset=['Target_Id']).copy()
edge_index_df['Target_Id'] = edge_index_df['Target_Id'].astype(int)
edge_list = edge_index_df[['New_Id', 'Target_Id']].rename(
    columns={'New_Id': 'source', 'Target_Id': 'target'}
)
print(edge_list.head())
edge_list.to_csv('../data/edge_list.txt', sep=' ', index=False, header=False)

   source  target
2       0   19591
2       0    2786
2       0   17782
2       0    7551
2       0   13262


In [37]:
asin_list = df_active[df_active['New_Id'] == 0]['SimilarASINs'].values[0]
print(asin_list)

print(df_active[df_active['New_Id'] == 29887][['ASIN']])
print(df_active[df_active['New_Id'] == 4212][['ASIN']])
print(df_active[df_active['New_Id'] == 27121][['ASIN']])
print(df_active[df_active['New_Id'] == 11394][['ASIN']])
print(df_active[df_active['New_Id'] == 20006][['ASIN']])

['0738700827', '1567184960', '1567182836', '0738700525', '0738700940']
Empty DataFrame
Columns: [ASIN]
Index: []
             ASIN
94493  0064431576
Empty DataFrame
Columns: [ASIN]
Index: []
              ASIN
258192  0834803399
              ASIN
464274  0805210644


# Ratings

## Print the Rating bins

In [38]:
discrete_counts = df_active['avg_rating'].value_counts().sort_index(ascending=False)

print(f"{'Rating':<10} | {'Count':<10} | {'Cumulative %'}")
print("-" * 35)

cumulative = 0
for rating, count in discrete_counts.items():
    cumulative += count
    pct = (cumulative / len(df_active)) * 100
    print(f"{rating:<10.2f} | {count:<10} | {pct:.2f}%")

Rating     | Count      | Cumulative %
-----------------------------------
5.00       | 6560       | 26.78%
4.50       | 9010       | 63.57%
4.00       | 5678       | 86.75%
3.50       | 2183       | 95.67%
3.00       | 772        | 98.82%
2.50       | 173        | 99.53%
2.00       | 78         | 99.84%
1.50       | 21         | 99.93%
1.00       | 17         | 100.00%


In [27]:
rating_map = {
    5.0: 0,
    4.5: 1,
    4.0: 2,
    3.5: 3,
    # Everything 3.0 and below gets grouped into Class 4
    3.0: 4, 2.5: 4, 2.0: 4, 1.5: 4, 1.0: 4 
}

df_active['label'] = df_active['avg_rating'].map(rating_map)

In [39]:
import numpy as np

labels = df_active['label'].round().astype(int)

counts = np.bincount(labels)
print("Class Distribution for Amazon Ratings:")
for i, count in enumerate(counts):
    percentage = (count / len(labels)) * 100
    print(f"Rating {i}: {count:>6} samples ({percentage:5.2f}%)")

Class Distribution for Amazon Ratings:
Rating 0:   6560 samples (26.78%)
Rating 1:   9010 samples (36.79%)
Rating 2:   5678 samples (23.18%)
Rating 3:   2183 samples ( 8.91%)
Rating 4:   1061 samples ( 4.33%)


In [54]:
df_active['label'].to_csv('../data/labels.csv', index=False, header=False)


# Generate NPZ

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from scripts.create_npz import create_npx

#create_npx.create_npz_dataset(node_features_file="../data/node_features.csv", labels_file='../data/labels.csv', edges_file='../data/edge_list.txt', num_splits=10, output_file_name='../data/amazon.npz')
create_npx.create_npz_dataset(node_features_file="../data/node_features.csv", labels_file='../data/labels.csv', edges_file='../data/edge_list.txt', num_splits=10, output_file_name='../data/amazon-fasttext.npz')


Verified Split 0 - Train: 14695, Val: 4898, Test: 4899
no of nodes 24492
no of features per node 300
no of node labels, should matach no of nodes 24492
no of edges 113276


In [71]:
import os

#file_path = '../data/amazon.npz'
file_path = '../data/amazon-fasttext.npz'
size_bytes = os.path.getsize(file_path)
size_mb = size_bytes / (1024 * 1024)

print(f"File Size: {size_mb:.2f} MB")

File Size: 30.65 MB


## Quick Check

In [72]:
import numpy as np

#data = np.load('../data/amazon.npz')
data = np.load('../data/amazon-fasttext.npz')
print("Keys in NPZ:", list(data.keys()))

# Expected keys usually include 'node_features', 'edges', 'labels', etc.
for key in data.keys():
    print(f"{key} shape: {data[key].shape}")

Keys in NPZ: ['node_features', 'node_labels', 'edges', 'train_masks', 'val_masks', 'test_masks']
node_features shape: (24492, 300)
node_labels shape: (24492,)
edges shape: (113276, 2)
train_masks shape: (10, 24492)
val_masks shape: (10, 24492)
test_masks shape: (10, 24492)


In [73]:
import numpy as np
data = np.load('../data/amazon.npz')
labels = data['node_labels']

# See which labels actually have nodes
unique, counts = np.unique(labels, return_counts=True)
print(dict(zip(unique, counts)))

{np.int64(0): np.int64(6560), np.int64(1): np.int64(9010), np.int64(2): np.int64(5678), np.int64(3): np.int64(2183), np.int64(4): np.int64(1061)}


In [74]:
counts = np.bincount(labels)
total_nodes = len(labels)
for i, count in enumerate(counts):
    percentage = (count / total_nodes) * 100
    print(f"Class {i}: {count:>6} samples ({percentage:5.2f}%)")


Class 0:   6560 samples (26.78%)
Class 1:   9010 samples (36.79%)
Class 2:   5678 samples (23.18%)
Class 3:   2183 samples ( 8.91%)
Class 4:   1061 samples ( 4.33%)
